# Georgia Bid Readiness: complete the DE warehouse
Run on the **school CPU**, in order. This extends your existing project and database.
No GPU, model inference, or paid API is used. Read `../README_DE_COMPLETION.md` first.
Stop at any error and share the output. Do not skip failed cells.


In [1]:
from pathlib import Path
import sys, os, subprocess, json, venv
PROJECT = Path('/home/jovyan/project/georgia-bid-readiness')
required = ['requirements-de.txt', 'scripts/run_de_pipeline.py',
 'src/transformation/requirement_loader.py', 'src/transformation/review_importer.py',
 'tests/test_requirement_loader.py', 'tests/test_canonical_requirements.py',
 'tests/test_review_importer.py', 'tests/fixtures/requirement_output_schema.json']
missing = [name for name in required if not (PROJECT / name).is_file()]
assert not missing, f'Missing prerequisite files: {missing}'
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Notebook Python:', sys.executable)


Project: /home/jovyan/project/georgia-bid-readiness
Notebook Python: /opt/conda/bin/python


## 1. Create an isolated DE environment
Installation needs internet access and may take several minutes. Existing GPU and dashboard packages are unchanged.

In [2]:
ENV = PROJECT / '.venv-de'
PY = ENV / 'bin/python'
if not PY.exists():
    venv.EnvBuilder(with_pip=True).create(ENV)
subprocess.run([str(PY), '-m', 'pip', 'install', '--no-cache-dir', '-r', 'requirements-de.txt'], check=True)
subprocess.run([str(PY), 'scripts/configure_de_gitignore.py'], check=True)
def run(*args, env=None):
    subprocess.run([str(PY), *args], cwd=PROJECT, env=env, check=True)
run('-c', "import sys,importlib.metadata as m; print(sys.version); print({x:m.version(x) for x in ['dbt-core','dbt-postgres','psycopg','pytest']})")


  Obtaining dependency information for dbt-core==1.9.8 from https://files.pythonhosted.org/packages/e5/3c/b2d36f6d8e551f0711aadb6d857f87da8e3ca4e1462eccd0e82f5433af87/dbt_core-1.9.8-py3-none-any.whl.metadata
  Obtaining dependency information for dbt-postgres==1.9.1 from https://files.pythonhosted.org/packages/d1/c7/7eb30411f81fd299410e7a4cebc73a60b8320179d032f2efe4bd378d7bae/dbt_postgres-1.9.1-py3-none-any.whl.metadata
  Obtaining dependency information for dbt-adapters==1.16.3 from https://files.pythonhosted.org/packages/c0/54/36267e4067d91bbafed68cc2c524cee8532021c880e7df53c396ea0385d8/dbt_adapters-1.16.3-py3-none-any.whl.metadata
  Obtaining dependency information for dbt-common==1.27.1 from https://files.pythonhosted.org/packages/b7/e7/b504d0814a99c0f33ea5d5735692025469dee01468ac1930c963f215635f/dbt_common-1.27.1-py3-none-any.whl.metadata
  Obtaining dependency information for psycopg[binary]==3.3.3 from https://files.pythonhosted.org/packages/c8/5b/181e2e3becb7672b502f0ed7f16ed73


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: /home/jovyan/project/georgia-bid-readiness/.venv-de/bin/python -m pip install --upgrade pip


Git exclusions ready.
3.11.5 | packaged by conda-forge | (main, Aug 27 2023, 03:34:09) [GCC 12.3.0]
{'dbt-core': '1.9.8', 'dbt-postgres': '1.9.1', 'psycopg': '3.3.3', 'pytest': '9.1.1'}


## 2. Check existing database and back it up
PostgreSQL must already be running. If connection fails, use the startup commands in the README. Do not run initdb.

In [3]:
run('scripts/run_de_pipeline.py', 'check')
run('-c', "import sys;sys.path.insert(0,'scripts');from de_backup import database_backup;from pathlib import Path;print(database_backup(Path('../georgia-bid-readiness-backups/de_completion'),verify=False))")


Run report: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T191448017697Z/run_report.json
{'dump': '../georgia-bid-readiness-backups/de_completion/gbr_20261025T191459029968Z.dump', 'source_tables': {'core.amendment_relation_reviews': {'rows': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'core.amendment_relations': {'rows': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'core.canonical_assertions': {'rows': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'core.canonical_requirements': {'rows': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'core.document_observations': {'rows': 27, 'sha256': '5c262caba61a02e87c6b20cd5e5cf08c4935156ad1715a7def7372e539a91ea7'}, 'core.document_versions': {'rows': 15, 'sha256': 'b18880a091bb3e74875e4b55b46ba5e829d381b905c718b80aa0d67712035e51'}, 'core.documents': {'rows': 15, 'sha256': '3260b80f71f8

## 3. Apply the additive migration and verify a repeat
Adds split-draft storage. The second execution must verify the existing migration checksum.

In [4]:
run('scripts/apply_de_completion_schema.py')
run('scripts/apply_de_completion_schema.py')


Migration 004 applied successfully.
Migration 004 already applied; checksum verified.


## 4. Run the database regression tests
Expected: **40 passed**. These tests create synthetic records inside transactions and roll them back.

In [5]:
test_env = os.environ.copy()
test_env.update(GBR_RUN_DB_TESTS='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1')
run('-m', 'pytest', 'tests/test_requirement_loader.py',
    'tests/test_canonical_requirements.py', 'tests/test_review_importer.py',
    'tests/test_de_completion.py', '-v', env=test_env)


============================= test session starts ==============================
platform linux -- Python 3.11.5, pytest-9.1.1, pluggy-1.7.0 -- /home/jovyan/project/georgia-bid-readiness/.venv-de/bin/python
cachedir: .pytest_cache
rootdir: /home/jovyan/project/georgia-bid-readiness
collecting ... collected 40 items

tests/test_requirement_loader.py::test_preflight_preserves_invalid_and_empty_outputs PASSED [  2%]
tests/test_requirement_loader.py::test_tampered_prompt_rejected PASSED   [  5%]
tests/test_requirement_loader.py::test_tampered_input_rejected PASSED    [  7%]
tests/test_requirement_loader.py::test_saved_flags_not_trusted PASSED    [ 10%]
tests/test_requirement_loader.py::test_saved_messages_not_trusted PASSED [ 12%]
tests/test_requirement_loader.py::test_missing_job_rejected PASSED       [ 15%]
tests/test_requirement_loader.py::test_preview_rolls_back PASSED         [ 17%]
tests/test_requirement_loader.py::test_repeat_load_and_bad_evidence PASSED [ 20%]
tests/test_requiremen

## 5. Preview without inserting groups or splits
Inspect the counts. First execution expects 13 groups, 14 assertions, 4 split drafts and 6 origin links. Repeats report reused records.

In [6]:
run('scripts/run_de_pipeline.py', 'preview')


Run report: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T191551937339Z/run_report.json


## 6. Run the complete warehouse refresh
Commits approved groupings and **unreviewed** split drafts, builds/tests dbt models, generates reports, verifies a database restore and creates a project backup. This can take several minutes. Detailed logs are written under reports/de_runs.

In [7]:
run('scripts/run_de_pipeline.py', 'run')


DE refresh, dbt tests, and restore verification passed.
Run report: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T191552943615Z/run_report.json


## 7. Check repeat safety and the completed report

In [8]:
result = subprocess.run([str(PY), 'scripts/complete_reviewed_requirements.py', '--commit'], cwd=PROJECT, check=True, capture_output=True, text=True)
repeat = json.loads(result.stdout)
assert not any(v for k,v in repeat['counts'].items() if k.endswith('_inserted')), repeat
print('Repeat load: no new records.', repeat)
reports = sorted((PROJECT / 'reports/de_runs').glob('*/run_report.json'))
full = [p for p in reports if json.loads(p.read_text())['mode'] == 'run']
assert full, 'No full run report found'
REPORT = full[-1]
summary = json.loads(REPORT.read_text())
assert summary['status'] == 'complete', summary
print('Completed:', REPORT)
for stage in summary['stages']:
    print(stage['name'], stage['status'])


Repeat load: no new records. {'mode': 'committed', 'counts': {'groups_inserted': 0, 'groups_reused': 13, 'assertions_inserted': 0, 'assertions_reused': 14, 'split_drafts_inserted': 0, 'split_drafts_reused': 4, 'split_origins_inserted': 0, 'split_origins_reused': 6}}
Completed: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T191552943615Z/run_report.json
preflight ok
before_change_backup ok
migration_004 ok
reviewed_requirements ok
dbt_build ok
dbt_docs ok
restore_verification ok
project_archive ok


In [9]:
run('-c', """
import sys
sys.path.insert(0,'scripts')
from de_common import settings
import psycopg
expected={'solicitations':2,'documents':15,'page_extractions':47,
 'model_responses':14,'requirement_candidates':32,'requirement_reviews':32,
 'canonical_requirements':13,'canonical_assertions':14,
 'split_requirements':4,'split_requirement_origins':6,'split_requirement_reviews':0}
from psycopg import sql
with psycopg.connect(**settings()) as conn:
 for table,count in expected.items():
  actual=conn.execute(sql.SQL('SELECT count(*) FROM core.{}').format(sql.Identifier(table))).fetchone()[0]
  print(table,actual,'PASS' if actual==count else 'CHANGED: inspect expected baseline')
  assert actual==count, (table,actual,count)
""")


solicitations 2 PASS
documents 15 PASS
page_extractions 47 PASS
model_responses 14 PASS
requirement_candidates 32 PASS
requirement_reviews 32 PASS
canonical_requirements 13 PASS
canonical_assertions 14 PASS
split_requirements 4 PASS
split_requirement_origins 6 PASS
split_requirement_reviews 0 PASS


The expected counts above describe the current two-solicitation pilot. If you deliberately add new sources/reviews later, revise that baseline after inspecting the change; do not delete records to satisfy it.

In [10]:
from IPython.display import display, FileLink, HTML
report_dir = REPORT.parent
display(HTML((report_dir / 'warehouse_summary.html').read_text()))
for name in ['run_report.json', 'warehouse_summary.html', 'warehouse_summary.json']:
    display(FileLink('../' + str((report_dir/name).relative_to(PROJECT))))
print('Backups:', PROJECT.parent / 'georgia-bid-readiness-backups/de_completion')
print('Download the verified .dump, its verification JSON, the project ZIP, and the final run report.')


solicitation_id,commerce-ga-25-012
agency_name,"City of Commerce, Georgia"
solicitation_number,25-012
title,Janitorial Services for City of Commerce Municipal Buildings
documents,12
extraction_records,28
extraction_failures,0
unresolved_page_flags,0
model_calls,0
invalid_responses,0
candidates,0


/home/jovyan/project/reports/de_runs/20261025T191552943615Z/run_report.json

/home/jovyan/project/reports/de_runs/20261025T191552943615Z/warehouse_summary.html

/home/jovyan/project/reports/de_runs/20261025T191552943615Z/warehouse_summary.json

Backups: /home/jovyan/project/georgia-bid-readiness-backups/de_completion
Download the verified .dump, its verification JSON, the project ZIP, and the final run report.


## 8. Save to GitHub after successful verification
Run in the project terminal. Inspect the staged list before committing; generated backups and virtual environments must not be included.

```bash
git add .gitignore .github/workflows/de-completion.yml README_DE_COMPLETION.md requirements-de.txt
git add dbt_gbr configs/de_completion_plan.json configs/de_completion_plan.sha256
git add warehouse/ddl/004_split_requirements.sql notebooks/10_de_completion_dbt.ipynb
git add scripts/apply_de_completion_schema.py scripts/de_common.py scripts/de_backup.py
git add scripts/complete_reviewed_requirements.py scripts/run_de_pipeline.py scripts/configure_de_gitignore.py
git add tests/test_de_completion.py tests/seed_dbt_ci.py reports/de_package_validation
git diff --cached --stat
git commit -m "Add dbt warehouse models, reviewed grouping and verified DE refresh"
git push
```

Check the new **DE completion** GitHub workflow. Passing locally does not prove that GitHub or the school environment passed.

The four proposed split requirements remain pending human review. This notebook does not mark them approved or invent amendment relationships. The broader product and model-evaluation work continues after this DE milestone.


In [11]:
from pathlib import Path
from IPython.display import display, HTML
from urllib.parse import quote
import os
import json

folder = Path(
    "/home/jovyan/project/georgia-bid-readiness/"
    "reports/de_runs/20261025T191552943615Z"
)

prefix = os.environ.get(
    "JUPYTERHUB_SERVICE_PREFIX",
    "/user/ishittu1@students.kennesaw.edu/"
).rstrip("/")

for name in [
    "run_report.json",
    "warehouse_summary.html",
    "warehouse_summary.json",
]:
    path = folder / name
    assert path.is_file(), f"Missing file: {path}"

    relative = path.relative_to("/home/jovyan").as_posix()
    url = f"{prefix}/files/{quote(relative, safe='/')}"
    display(HTML(f'<a href="{url}" target="_blank">{name}</a>'))

report = json.loads((folder / "run_report.json").read_text())
print("Pipeline status:", report["status"])

Pipeline status: complete
